# 13 · Contrastive encoder for known-condition recognition (Stage R3)

## Purpose
Does training the CNN-6 encoder with a **supervised contrastive objective** (SupCon), instead of cross-entropy alone,
improve recognition of the ten known conditions across operating settings and speeds, and separate the confusable pairs
(imbalance ↔ vertical misalignment, horizontal ↔ vertical misalignment)? Protocol declared in CLAUDE.md §7 (Stage R3)
before any training; exploratory, development data only; reserved tests sealed.

The encoder is **CNN-6 unchanged**; only the objective differs. Stage A trains it with SupCon, where a positive pair is two
windows of the same class from **different recordings** (same-recording pairs are excluded entirely, one view per window),
in class-balanced batches that favour different folders. Stage B trains the CNN-6 head on the **frozen** encoder with the
CNN-6 recipe. Augmentations are exactly CNN-6's (gain ±1 dB, time shift ±2 frames, one time mask of 0–4 frames); none
touches speed or the order axis. Training ran in `scripts/33_train_contrastive.py`; this notebook reads saved outputs.

**Ceiling.** On these development folds CNN-6 misclassifies 8 / 11 / 4 of 804 recordings (seeds 42 / 43 / 44). Any difference below therefore rests on a handful of recordings. The development pool runs below 40 Hz and excludes the largest severity of each family, so these folds cannot measure the development-to-final gap.

## Inputs
`metadata.csv`, `results/dev_folds.csv` (K = 3 folder-grouped folds); `configs/contrastive.yaml`;
`results/research/r3_contrastive.json`, `r3_per_fold.csv`, `r3_confusion.csv` (from `33_train_contrastive.py evaluate`);
`results/contrastive/supcon6_s<seed>_dev_train.json` (pair statistics, training time); saved logits and embeddings
`results/scores/supcon6_s<seed>_dev_*` and the CNN-6 baselines `results/scores/cnn6_dev*`, `cnn6_s43_dev_*`, `cnn6_s44_dev_*`.
Config and split hashes are printed below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the configuration, the development pool and the saved R3 results, prints the config and split hashes, and checks the pool size (804 recordings over 3 folds). It stops with an error if a count differs.</div>

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.config import load_config, config_sha256
from ml.contrastive import load_contrastive_config
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, RunPaths
from ml.splits import development_pool, split_hash

ROOT = SYSTEM_CODE_DIR
R = ROOT / "results"
cfg = load_config(DEFAULT_CONFIG)
kc = load_contrastive_config(ROOT / "configs/contrastive.yaml")
meta = pd.read_csv(ROOT / "metadata.csv")
pool = development_pool(meta, cfg.development.allowed_closed, cfg.development.reserved_columns).merge(
    pd.read_csv(R / cfg.development.folds_file)[["rec_id", "dev_fold"]], on="rec_id")
assert len(pool) == 804, "development pool size differs: stop and ask"
print("config sha256:", config_sha256(cfg)[:16], "| split hash:", split_hash(meta)[:16])
print("recordings per fold:", pool.groupby("dev_fold").size().to_dict())
res = json.loads((R / "research/r3_contrastive.json").read_text())
pf = pd.read_csv(R / "research/r3_per_fold.csv")
conf = pd.read_csv(R / "research/r3_confusion.csv")
SEEDS = (42, 43, 44)
train = {s: json.loads(RunPaths("contrastive", "supcon6", s).result("train").read_text()) for s in SEEDS}
print("contrastive settings:", {k: v for k, v in kc.model_dump().items() if k in ("temperature", "recs_per_class", "epochs", "head_epochs", "lr")})

## Realised pair statistics (design constraint 2)
Every positive pair used by the loss, counted over all batches of stage A: how many came from different recordings (must
be all of them), different folders and different speeds (|Δ measured speed| ≥ 1 Hz). Normal has a single folder, so its
positives can never cross folders.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> tabulates, per seed and fold, the number of positive pairs the loss used and the share from different recordings, different folders and different speeds, with the median speed difference. A same-recording share above zero would break the declared rule.</div>

In [ ]:
ps = pd.DataFrame([{"seed": s, "fold": f["fold"], **f["pair_stats"]} for s in SEEDS for f in train[s]["folds"]])
assert (ps.same_recording_positives == 0).all() and (ps.frac_different_recordings == 1).all()
print(ps[["seed", "fold", "positive_pairs", "frac_different_recordings", "frac_different_folders", "frac_different_speeds",
          "median_abs_speed_diff_hz"]].round(3).to_string(index=False))

## Training cost
Stage A (contrastive) and stage B (head) seconds per seed, against CNN-6's training seconds on the same folds and seeds.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the measured training time of each seed (stage A, stage B, total) next to CNN-6's, and the time estimate that was projected after the first fold (the declared guard: stop above 60 minutes).</div>

In [ ]:
cost = res["cost"]
t = pd.DataFrame({s: {"stage A (s)": cost["train_seconds"][str(s)]["stage_a"], "stage B (s)": cost["train_seconds"][str(s)]["stage_b"],
                      "CNN-6 (s)": cost["cnn6_train_seconds"][str(s)]} for s in SEEDS}).T
t["contrastive total (min)"] = (t["stage A (s)"] + t["stage B (s)"]) / 60
print(t.round(1))
est = json.loads((R / "research/r3_time_estimate.json").read_text())
print(f"projected before continuing: {est['projected_total_minutes']:.0f} min (guard {est['guard_minutes']} min); "
      f"actual: {t['contrastive total (min)'].sum():.0f} min")

## Primary: per-fold macro-F1 and balanced accuracy, contrastive vs CNN-6
Per seed: the three fold values, their mean and spread (§5 rule 13). The difference Δ_s of the fold means drives the
declared reading.

**Ceiling.** On these development folds CNN-6 misclassifies 8 / 11 / 4 of 804 recordings (seeds 42 / 43 / 44). Any difference below therefore rests on a handful of recordings. The development pool runs below 40 Hz and excludes the largest severity of each family, so these folds cannot measure the development-to-final gap.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> pivots the per-fold results into one table per metric (rows: seed and fold; columns: CNN-6 and contrastive), then prints the fold means and ranges per seed and the difference contrastive − CNN-6 per seed, for the ten-class and six-class views.</div>

In [ ]:
for m in ("class10_macro_f1", "class10_balanced_accuracy", "class6_macro_f1", "class6_balanced_accuracy"):
    t = pf.pivot_table(index=["seed", "fold"], columns="model", values=m)
    g = pf.groupby(["seed", "model"])[m].agg(["mean", "min", "max"]).unstack("model")
    print(f"--- {m}"); print(t.round(4).to_string()); print(g.round(4).to_string())
print("Δ class10 macro-F1 (fold mean) per seed:", {k: round(v, 4) for k, v in res["fold_mean_delta_class10_macro_f1"].items()})
print("Δ class6  macro-F1 (fold mean) per seed:", {k: round(v, 4) for k, v in res["fold_mean_delta_class6_macro_f1"].items()})
print("folds where contrastive ≥ CNN-6:", res["folds_where_supcon_ge_cnn6"])

## Two sources of uncertainty, reported separately
**Seed spread**: the standard deviation over seeds 42/43/44 of the fold-mean metric (training-run variation).
**Sampling uncertainty**: per seed, the paired mixed-unit bootstrap of the pooled macro-F1 difference (fault folders
resampled within class, normal recordings individually; the Stage 3A draws). They are never merged.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the mean and standard deviation over seeds of each model's fold-mean metrics, and, per seed, the paired bootstrap difference contrastive − CNN-6 in pooled macro-F1 with its 95 % interval and the share of draws above zero.</div>

In [ ]:
for m, d in res["seed_spread"].items():
    print(m, {k: f"{v['mean']:.4f} ± {v['std_over_seeds']:.4f}" for k, v in d.items()})
for s, d in res["paired_bootstrap"].items():
    print(f"seed {s}: class10 {d['class10']['diff']:+.4f} [{d['class10']['ci'][0]:+.4f}, {d['class10']['ci'][1]:+.4f}] "
          f"P(>0) {d['class10']['p_positive']:.2f} | class6 {d['class6']['diff']:+.4f} [{d['class6']['ci'][0]:+.4f}, {d['class6']['ci'][1]:+.4f}]")

## Per-class recall and the error count

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows the recall of each of the ten classes (share of its recordings recognised correctly) for each model and seed, and the total number of misclassified recordings out of 804.</div>

In [ ]:
rec = pd.DataFrame({k: v["per_class_recall"] for k, v in res["models"].items()})
print(rec.round(3).to_string())
print("misclassified recordings:", {k: v["errors"] for k, v in res["models"].items()})

## Confusion matrices and the named pairs
Imbalance ↔ vertical misalignment and horizontal ↔ vertical misalignment are called out (recordings, both directions, per
seed). The figure sums the three seeds (2,412 recording decisions per model); diagonal cells are omitted so the errors are
visible, and the named-pair cells are outlined.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the named-pair confusions per model and seed and the declared pair reading, then draws the off-diagonal confusion counts summed over the three seeds for CNN-6 and for the contrastive model side by side, with the named-pair cells outlined in orange, and saves the figure for the report.</div>

In [ ]:
for name, d in res["pair_reading"].items():
    print(name, "| CNN-6 per seed:", d["cnn6_per_seed"], "| contrastive:", d["supcon6_per_seed"],
          "| separates better:", d["separates_better"], d["note"])
for k, v in res["models"].items():
    print(k, v["pair_errors"])
short = {c: c.replace("horizontal_misalignment", "horiz_mis").replace("vertical_misalignment", "vert_mis") for c in CLASS10}
pairs = [("imbalance", "vertical_misalignment"), ("horizontal_misalignment", "vertical_misalignment")]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))
for ax, m, title in zip(axes, ("cnn6", "supcon6"), ("CNN-6 (cross-entropy)", "Contrastive encoder + frozen head")):
    M = conf[conf.model == m].pivot_table(index="true", columns="pred", values="recordings", aggfunc="sum").reindex(
        index=CLASS10, columns=CLASS10).fillna(0).to_numpy()
    off = M.copy(); np.fill_diagonal(off, np.nan)
    ax.imshow(np.nan_to_num(off), cmap="Reds", vmin=0, vmax=max(1, np.nanmax(off)))
    for i in range(10):
        for j in range(10):
            if i != j and M[i, j]:
                ax.text(j, i, int(M[i, j]), ha="center", va="center", fontsize=9)
    for a, b in pairs:
        for i, j in ((CLASS10.index(a), CLASS10.index(b)), (CLASS10.index(b), CLASS10.index(a))):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="#ff8c00", lw=2))
    ax.set_xticks(range(10), [short[c] for c in CLASS10], rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(10), [short[c] for c in CLASS10], fontsize=8)
    ax.set_xlabel("predicted class"); ax.set_ylabel("true class")
    ax.set_title(f"{title}: {int(np.nansum(off))} errors (3 seeds × 804 recordings)", fontsize=10)
fig.suptitle("Off-diagonal recording confusions, development folds, summed over seeds 42/43/44 (named pairs outlined)", fontsize=10)
fig.tight_layout()
fig.savefig(R / "figures/r3_confusion.png", dpi=150)
plt.show()

## Operating settings and speeds
Accuracy per folder (a folder is one operating setting: a severity level of one fault) and per speed band. Only folders
where either model errs in some seed are shown.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> lists, for every folder where at least one model misclassifies a recording in some seed, the per-seed accuracy of both models; then the accuracy in three speed bands (below 20 Hz, 20–30 Hz, 30 Hz and above) per model and seed.</div>

In [ ]:
fa = pd.DataFrame({k: v["per_folder_accuracy"] for k, v in res["models"].items()})
print(fa[(fa < 1).any(axis=1)].round(3).to_string())
sb = pd.DataFrame({k: {b: f"{d['accuracy']:.4f} (n={d['n']})" for b, d in v["per_speed_band_accuracy"].items()}
                   for k, v in res["models"].items()})
print(sb.to_string())

## Embedding view (seed 42)
Recording-level embeddings (mean of the 9 window embeddings, each scored by its own fold's model) projected to 2-D with
t-SNE, CNN-6 vs contrastive. Folds are embedded by different models, so distances across folds are not comparable; the
picture is qualitative only.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the saved 128-d window embeddings of both seed-42 models, averages them per recording, projects each set to two dimensions with t-SNE (fixed random state) and plots the recordings coloured by class, with the named pairs in saturated colours. The figure is saved for the report; it is illustrative, not evidence.</div>

In [ ]:
from sklearn.manifold import TSNE
def rec_emb(path):
    z = np.load(path)
    df = pd.DataFrame(z["embedding"]); df["rec_id"] = z["rec_id"]
    return df.groupby("rec_id").mean().loc[sorted(pool.rec_id)]
cls = pool.set_index("rec_id").class10.loc[sorted(pool.rec_id)]
colours = dict(zip(CLASS10, plt.cm.tab10.colors))
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, (title, p) in zip(axes, (("CNN-6, seed 42", R / "scores/cnn6_dev_embeddings.npz"),
                                 ("Contrastive, seed 42", RunPaths("contrastive", "supcon6", 42).embeddings()))):
    e = rec_emb(p)
    xy = TSNE(n_components=2, random_state=42, init="pca", perplexity=30).fit_transform(e.to_numpy())
    for c in CLASS10:
        m = (cls == c).to_numpy()
        ax.scatter(xy[m, 0], xy[m, 1], s=9, color=colours[c], label=c, alpha=0.8)
    ax.set_title(title); ax.set_xlabel("t-SNE 1 (arbitrary units)"); ax.set_ylabel("t-SNE 2 (arbitrary units)")
axes[1].legend(fontsize=7, markerscale=2, loc="best")
fig.suptitle("Recording embeddings (mean of 9 windows), development pool; qualitative only", fontsize=10)
fig.tight_layout()
fig.savefig(R / "figures/r3_embedding.png", dpi=150)
plt.show()

## Inference latency
Median CPU milliseconds to score one recording (9 windows from the cached representation, one batch; the Stage 3A method),
both models on the same fold-0 recordings in the same process. The deployed contrastive model is a ScratchCNN with the same
architecture and parameter count as CNN-6.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints the measured CPU latency per recording of both models.</div>

In [ ]:
print(res["cost"]["cpu_latency_ms_per_recording"])

## Pre-declared reading
**Helps** only if (a) Δ_s > 0 for every seed, (b) mean Δ ≥ +0.003, (c) the contrastive per-fold macro-F1 ≥ CNN-6's in at
least 2 of 3 folds in every seed, (d) class6 mean Δ ≥ −0.003. **Does not help** if Δ_s ≤ 0 for every seed or mean Δ ≤
−0.003. Otherwise **too thin to tell**. The optional Q3 step runs only on "helps".

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints each declared criterion with its value and the resulting reading, and whether the optional Q3 step was run.</div>

In [ ]:
print({k: v for k, v in res["criteria"].items()})
print(f"mean Δ class10 {res['mean_delta_class10']:+.4f}, mean Δ class6 {res['mean_delta_class6']:+.4f}")
print("READING:", res["reading"], "| optional Q3 step:", res["optional_q3_step"])

## Findings

Exploratory development results (folder-grouped K = 3 folds, seeds 42 / 43 / 44). The development pool runs below 40 Hz and
excludes the largest severity of each family, so these folds cannot measure the development-to-final gap; CNN-6 errs on
8 / 11 / 4 of 804 recordings here, so every difference rests on a small number of recordings.

- **Pre-declared reading: "does not help".** The fold-mean class10 macro-F1 is lower than CNN-6's in every seed
  (Δ −0.018 / −0.010 / −0.015; mean −0.014, threshold for "does not help" −0.003). Contrastive per-fold macro-F1 was ≥
  CNN-6's in 0 / 1 / 0 of 3 folds. Six-class: Δ −0.021 / +0.003 / −0.011 (mean −0.010). No criterion for "helps" is met,
  so the optional Q3 (Mahalanobis) step was not run.
- **Seed spread and sampling uncertainty, separately.** Fold-mean class10 macro-F1 over seeds: CNN-6 0.989 ± 0.006,
  contrastive 0.975 ± 0.004 (std over seeds). Paired mixed-unit bootstrap per seed (contrastive − CNN-6, pooled class10
  macro-F1): −0.020 [−0.032, −0.008], −0.009 [−0.029, +0.003], −0.015 [−0.028, −0.004]. Misclassified recordings: 21 / 19 / 14
  against CNN-6's 8 / 11 / 4.
- **Named pairs: no better separation, slightly worse.** Imbalance ↔ vertical misalignment: CNN-6 0 / 0 / 0, contrastive
  0 / 1 / 1 (CNN-6 had no confusions, so only a worsening could be observed). Horizontal ↔ vertical misalignment: CNN-6
  2 / 0 / 0, contrastive 5 / 1 / 0. Most new errors are bearing confusions: underhang cage → overhang outer 13, overhang
  cage → overhang outer 10, overhang outer → overhang cage 8 (sums over seeds).
- **Across operating settings and speeds:** the losses sit in a few folders (overhang cage 20 g, overhang outer 20 g,
  horizontal misalignment 1 mm and 1.5 mm in seed 42) and in the < 20 Hz and ≥ 30 Hz bands (e.g. ≥ 30 Hz accuracy
  0.982 / 0.971 / 0.989 for CNN-6 vs 0.968 / 0.950 / 0.964); 20–30 Hz is unchanged (0.997 vs 0.993–0.997).
- **The pair rule held.** 5.0–5.2 million positive pairs per fold, 100 % from different recordings (0 same-recording
  positives), 53–54 % from different folders (normal has one folder; bearing classes have three) and 94–95 % from
  different speeds (median |Δ| 8.2 Hz). Positives were not chosen to differ in speed, and no augmentation touches speed.
- **Cost:** 26 min in total, as projected after the first fold (guard 60 min): 8.5–8.6 min per seed against CNN-6's
  3.0 min (stage A ≈ 147 s, stage B ≈ 23 s per fold). CPU latency per recording is unchanged in kind (same architecture):
  3.9 ms vs 4.4 ms for CNN-6 in the same process; the difference is measurement noise. **What this does and does not
  show:** this contrastive objective, with a frozen encoder and these fixed settings, did not improve known-condition
  recognition on these folds. It does not show that no contrastive or fine-tuned variant could help, and these folds
  cannot speak to the speed- and severity-extrapolation gap on the sealed tests.


## Outputs written
- By `scripts/33_train_contrastive.py`: `results/research/r3_contrastive.json`, `r3_per_fold.csv`, `r3_confusion.csv`,
  `r3_time_estimate.json`; `results/contrastive/supcon6_s<seed>_dev_train.json` and fold normalisation files;
  `results/scores/supcon6_s<seed>_dev_logits.parquet` / `_embeddings.npz`; weights in `artifacts/contrastive/` (gitignored).
- By this notebook: `results/figures/r3_confusion.png`, `results/figures/r3_embedding.png`.